In [ ]:
# Interactive Figure
%matplotlib widget 

In [ ]:
import time
import numpy as np
from matplotlib import pyplot as plt
from qualang_tools.units import unit
u = unit(coerce_to_integer=True)
from qm.qua import *
from qm import SimulationConfig
from qualang_tools.loops import from_array
import QM

# Send pulses and observe them

In [ ]:
# This block is using QUA directives which are compiled to the FPGA
with program() as prog:
    # Variable declaration
    adc_stream = declare_stream(adc_trace=True)
    # Start measurement
    measure('readout', 'scope', adc_stream=adc_stream)
    # Pulse sequence
    play('pulse','qubit')
    wait(24*u.ns)
    play('pulse','qubit')

    # Select data to be sent to the client 
    with stream_processing():
        adc_stream.input1().save('adc_1')
        adc_stream.input2().save('adc_2')

In [ ]:
# Run the code and fetch results
job = QM.Job(prog)

In [ ]:
# Fetch results
adc_1, adc_2 = job.get_results('adc_1','adc_2')

In [ ]:
# Plot
readout_len = 800 # in ns from the config
t = np.arange(readout_len)
fig,ax=plt.subplots()
ax.plot(t,adc_1,t,adc_2)

# Simulate QUA code

In [ ]:
plt.figure()
# Simulates the QUA program for the specified duration
simulation_config = SimulationConfig(duration=readout_len//4)  # In clock cycles = 4ns
# Simulate blocks python until the simulation is done
config = QM.get_config(full=True)
job = QM.qmm.simulate(config, prog, simulation_config)
# Get the simulated samples
samples = job.get_simulated_samples()
# Plot the simulated samples
samples.con1.plot()
plt.show()

# Display QM configuration

In [ ]:
QM.show_config()

# Exercises
- Link to the Quantum Machines documentation:  [https://docs.quantum-machines.co/latest/](https://docs.quantum-machines.co/latest/)
- Read the documentation on the [`play`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.play) command
- In this first sequence, you will learn how to play pulses with the QM. Two pulses have been preloaded in the machine, a constant pulse and a gaussian pulse. The pulses can be modified on the fly during execution. The amplitude can be multiplied by a number between -2 and 2, and the duration can be changed. If you do not specify a unit, the default duration unit is one clock cycle, which is 4 ns. The unit `u.ns`and `u.us` are convenient multipliers to perfom the conversion into clock cycles.
- The QM plays the different pulses in parallel on the different outputs but sequentially on the same output.

## Exercise 1: Phase Evolution
- Play two pulses one after the other with no waiting time. Are the traces continuous? Why?
- Dephase the second pulse by $\pi/2$ using a [`frame_rotation`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.frame_rotation) command applied to the `qubit` element. Add a delay between the pulses again and observe the phase evolution.

## Exercise 2: Pulse Timing
- Change the pulse duration to 100 ns using the `duration` keyword in the [`play`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.play) command.
- Modify the sequence to create two pulses separated by 500 ns with a [`wait`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.wait) command.

## Exercise 3: Arbitrary Pulses and Amplitude Tuning
- In the [`play`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.play) command, replace `pulse` with `gaussian` and observe the results (remove the `duration` parameter if present). You can look in the `config_00.py` file to see how arbitrary pulses are constructed.
- Modify the pulse amplitude using [`amp`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.amp) in the [`play`](https://docs.quantum-machines.co/latest/docs/API_references/qua/dsl_main/#for_docs.dsl.play) command.
- The default duration of the gaussian pulse is 200 ns. Expand it to a larger duration using the `duration` parameter.